# BRCA all omics benchmark
- benchmark for the brca dataset
- take
    - [x] mRNA
    - [x] CNA
    - [x] miRNA
    - [ ] DNA methylation data
- benchmark several prediction methods on them, using
    - [ ] early integration
    - [ ] late integration

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import polars as pl
import torch
import torch_geometric as pyg
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split

from bipartite_gnn.graph_visualizatons import visualize_graph, visualize_embeddings
from baseline_evals.feature_selection import variance_filtering, class_variational_selection
from baseline_evals.knn_eval import knn_eval
from baseline_evals.svm_eval import svm_eval
from baseline_evals.xgboost_eval import xgboost_eval
from baseline_evals.mlp_eval import mlp_eval

In [3]:
null_vals = ["NA"]
mrna = pl.read_csv("BRCA_PROCESSED_DATA/mrna.tsv", separator="\t", null_values=null_vals)
cna = pl.read_csv("BRCA_PROCESSED_DATA/cnvth.tsv", separator="\t", null_values=null_vals)
mirna = pl.read_csv("BRCA_PROCESSED_DATA/mirna.tsv", separator="\t", null_values=null_vals)

In [4]:
labels = pl.read_csv("BRCA_PROCESSED_DATA/labels.tsv", separator="\t")
le = LabelEncoder()
le.fit(labels["PAM50_mRNA_nature2012"].to_list())
y = le.transform(labels["PAM50_mRNA_nature2012"].to_list())

In [5]:
# ensure that the omic channels are alined with the labels and with each other
assert mrna.columns[1:] == cna.columns[1:] == mirna.columns[1:] == labels["sampleID"].to_list()

In [6]:
# TODO, join genes with identical features into a single vertex
def find_identical_rows(matrix):
    # Convert the matrix to a structured array
    structured_array = np.core.records.fromarrays(matrix.transpose())
    
    # Find unique rows and their corresponding labels
    _, labels = np.unique(structured_array, return_inverse=True)
    
    # Group indices by labels
    unique_labels, indices = np.unique(labels, return_index=True)
    identical_rows_indices = [np.where(labels == label)[0] for label in unique_labels]
    
    return identical_rows_indices

# Example usage
matrix = np.array([[1, 2], [3, 4], [1, 2], [5, 6], [3, 4]])
identical_rows_indices = find_identical_rows(matrix)

for indices in identical_rows_indices:
    print(f"Identical rows at indices: {indices}")

# identical_rows = find_identical_rows(cna_m)

Identical rows at indices: [0 2]
Identical rows at indices: [1 4]
Identical rows at indices: [3]


In [7]:
X_mrna = mrna[:,1:].to_numpy().T
X_cna = cna[:,1:].to_numpy().T
X_mirna = mirna[:,1:].to_numpy().T

X = np.hstack([X_mrna, X_cna, X_mirna])
X = X_mrna
X.shape

(483, 18975)

## Benchmarks

In [8]:
knn_eval(X, y, n_features=5000, test_size=0.2)

| KNN | 0.72 +/- 0.04 | 0.63 +/- 0.07 | 0.67 +/- 0.05 |
study.best_value=0.6672525425100214, study.best_params={'n_neighbors': 7}


In [17]:
svm_eval(X, y, n_trials=15, n_features_preselect=10000, n_features=1000, test_size=0.2, mode="linear")

Trial 1 / 15
| SVM | 0.80 +/- 0.05 | 0.79 +/- 0.05 | 0.80 +/- 0.05 |
Trial 2 / 15
Trial 3 / 15
| SVM | 0.80 +/- 0.05 | 0.80 +/- 0.05 | 0.81 +/- 0.05 |
Trial 4 / 15
Trial 5 / 15
Trial 6 / 15
| SVM | 0.81 +/- 0.05 | 0.80 +/- 0.05 | 0.81 +/- 0.05 |
Trial 7 / 15
| SVM | 0.81 +/- 0.05 | 0.81 +/- 0.04 | 0.82 +/- 0.05 |
Trial 8 / 15
| SVM | 0.81 +/- 0.05 | 0.81 +/- 0.05 | 0.82 +/- 0.04 |
Trial 9 / 15
Trial 10 / 15
Trial 11 / 15
Trial 12 / 15
| SVM | 0.82 +/- 0.04 | 0.81 +/- 0.04 | 0.82 +/- 0.04 |
Trial 13 / 15
Trial 14 / 15
Trial 15 / 15
| LIN SVM | 0.82 +/- 0.04 | 0.81 +/- 0.04 | 0.82 +/- 0.04 |
study.best_value=0.8221886926857238, study.best_params={'C': 0.0010512399433142698, 'class_weight': None}


{'acc': 0.8177721088435372,
 'f1_macro': 0.813039065612194,
 'f1_weighted': 0.8221886926857238,
 'acc_std': 0.0424187640477469,
 'f1_macro_std': 0.04175387725335554,
 'f1_weighted_std': 0.0404691081318716}

In [26]:
svm_eval(X, y, n_trials=20, n_features_preselect=5000, n_features=500, test_size=0.4, mode="rbf")

Trial 1 / 20


ValueError: when `importance_getter=='auto'`, the underlying estimator SVC should have `coef_` or `feature_importances_` attribute. Either pass a fitted estimator to feature selector or call fit before calling transform.

In [10]:
xgboost_eval(X, y, n_features=10000, n_evals=10, test_size=0.2)

0 / 50
ACC: [0.82474227 0.84536082 0.89690722 0.90721649 0.89690722 0.87628866
 0.87628866 0.88659794 0.92783505 0.83505155]
F1M: [0.82868669 0.8384615  0.8907259  0.91694454 0.88835282 0.85576165
 0.87405303 0.88089783 0.92386233 0.8234386 ]
F1W: [0.82087577 0.84554546 0.89636059 0.90711035 0.89865853 0.87726155
 0.87628866 0.88618142 0.92691926 0.83876563]
| XGBoost | 0.88 +/- 0.03 | 0.87 +/- 0.03 | 0.88 +/- 0.03 |
1 / 50
Pruning trial
ACC: [0.45360825 0.45360825 0.45360825 0.45360825 0.45360825 0.45360825
 0.45360825 0.         0.         0.        ]
F1M: [0.15602837 0.15602837 0.15602837 0.15602837 0.15602837 0.15602837
 0.15602837 0.         0.         0.        ]
F1W: [0.28310302 0.28310302 0.28310302 0.28310302 0.28310302 0.28310302
 0.28310302 0.         0.         0.        ]
2 / 50
Pruning trial
ACC: [0.78350515 0.77319588 0.7628866  0.82474227 0.83505155 0.83505155
 0.82474227 0.         0.         0.        ]
F1M: [0.78386208 0.74498992 0.74882094 0.82546407 0.7994489  0.79

KeyboardInterrupt: 

| XGBoost | 0.86 +/- 0.05 | 0.86 +/- 0.05 | 0.86 +/- 0.05 |
| XGBoost | 0.88 +/- 0.03 | 0.88 +/- 0.03 | 0.88 +/- 0.03 |

In [11]:
mlp_eval(X, y, n_trials=15, n_evals=6, n_features=6000, val_test_size=0.15)

Trial 0 / 15
Eval 1 / 6


/home/lubojjan/micromamba/envs/diploma/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:75: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default


Eval 2 / 6
Eval 3 / 6
Eval 4 / 6
Eval 5 / 6
Eval 6 / 6
[0.75937498 0.83749998 0.953125   0.77499998 0.67500001 0.72187501]
[0.50912356 0.58905229 0.96405229 0.746337   0.43102919 0.69444444]
[0.75649246 0.83698938 0.95261438 0.77923535 0.69030338 0.78333333]
{'acc': 0.7869791587193807, 'f1_macro': 0.6556731275051876, 'f1_weighted': 0.7998280459669084, 'acc_std': 0.0892936770132677, 'f1_macro_std': 0.17378188921621968, 'f1_weighted_std': 0.08092629247747436}
Trial 1 / 15
Eval 1 / 6
Eval 2 / 6
Eval 3 / 6
Eval 4 / 6
Pruning trial after 3 evals, cause 0.6161768264709441 < 0.7998280459669084
[0.7027027  0.75675678 0.62162161 0.64864862 0.         0.        ]
[0.55357143 0.57222851 0.39652778 0.40763889 0.         0.        ]
[0.66023166 0.69603155 0.49226727 0.51268769 0.         0.        ]
Trial 2 / 15
Eval 1 / 6
Eval 2 / 6
Eval 3 / 6
Eval 4 / 6
Eval 5 / 6
Eval 6 / 6
[0.72972971 0.83783782 0.72972971 0.7027027  0.75675678 0.75675678]
[0.57383901 0.65404412 0.58333333 0.56944444 0.57628205

Exception ignored in: <function _releaseLock at 0x79e345bbcea0>
Traceback (most recent call last):
  File "/home/lubojjan/micromamba/envs/diploma/lib/python3.12/logging/__init__.py", line 243, in _releaseLock
    def _releaseLock():
    
KeyboardInterrupt: 


{'acc': 0.8375, 'f1_macro': 0.8159961268328166, 'f1_weighted': 0.8367642441789332, 'acc_std': 0.09275513799784894, 'f1_macro_std': 0.07937400285564268, 'f1_weighted_std': 0.09272688443586957}


In [7]:
from baseline_evals.mlp_eval import MLP

mlp_best = MLP(3000, 4, 47, [47, 70], 0.4)
mlp_best.load_state_dict(torch.load("mlp_best_model.pth"))

<All keys matched successfully>

In [8]:
mlp_best.hidden_layers[0].weight.shape, mlp_best.hidden_layers[0].bias.shape
mlp_best.proj.weight.shape, mlp_best.proj.bias.shape

(torch.Size([47, 3000]), torch.Size([47]))

In [28]:
feat_importances = mlp_best.proj.weight.sum(dim=0)

In [ ]:
mlp_best.proj.we

In [29]:
(torch.round(feat_importances, 4) == 0).sum()

tensor(0)

In [1]:
from matplotlib import pyplot as plt

plt.hist(mlp_best.proj.weight.sum(dim=0).detach().numpy(), bins=100)

NameError: name 'mlp_best' is not defined

study.best_value=2.5192529195151394

study.best_params={'l1_lambda': 0.0015844617502738152, 'batch_sz': 64, 'proj_dim': 47, 'dropout': 0.4237635831392694, 'hidden_channels': 70}

| MLP | 0.85 +/- 0.02 | 0.82 +/- 0.04 | 0.85 +/- 0.03 |

w/o l1 reg on the first layer

| MLP | 0.77 +/- 0.02 | 0.73 +/- 0.05 | 0.76 +/- 0.02 |

In [ ]:
# bipartite graph
